In [ ]:
# Copyright 2026 50Hertz Transmission GmbH and Elia Transmission Belgium SA/NV
#
# This Source Code Form is subject to the terms of the Mozilla Public License, v. 2.0.
# If a copy of the MPL was not distributed with this file,
# you can obtain one at https://mozilla.org/MPL/2.0/.
# Mozilla Public License, version 2.0


# Example 5: the whole pipeline, called by hand, AC-validated end to end

`example3` runs a grid through the *pipeline* helpers (`run_preprocessing`, `run_dc_optimization_stage`,
`perform_ac_analysis`, `run_pipeline`). This notebook takes the same journey — import, DC preprocess,
DC optimize, AC validate — but with the packages called **directly**, the way `example4` inspects each
hand-over. There is no pipeline orchestration and no results-folder scaffolding: just the functions.

The goal is the last stage: take the topology the optimizer found and **AC-validate** it, then show, side
by side, whether every N-1 loadflow converged on the **initial** topology and on the **found** one.

| Stage | Package | Direct call | Produces |
|---|---|---|---|
| 1. Import | `toop_engine_importer` | `preprocessing.convert_file` | `nminus1_definition.json`, `action_set.*`, the processed grid |
| 2. DC preprocessing | `toop_engine_dc_solver` | `load_grid` | `StaticInformation`, `NetworkData` |
| 3. Topology optimization | `toop_engine_topology_optimizer` | `initialize_optimization` + `run_epoch` | candidate topologies (`actions`, `disconnections`) |
| 4. AC validation | `toop_engine_contingency_analysis` | `apply_topology` + `get_ac_loadflow_results` | `LoadflowResultsPolars` — initial vs found |

We import the grid **network-derived**, exactly as `example3` does: the importer derives the masks, the
`action_set` and an N-1 definition from the grid itself. For the AC validation in Stage 4 you can choose the
N-1 definition: either that **mask-derived** one, or the **input N-1 definition** written for this grid
(`create_complex_grid_nminus1_definition` from `toop_engine_grid_helpers`), validated against the processed grid. Set
`USE_INPUT_NMINUS1_DEFINITION` below.


In [ ]:
import logging
import tempfile
import time
from pathlib import Path

import polars as pl
import pypowsybl
import structlog

# Keep warnings, which is where the contract diagnostics surface; silence info-level chatter.
structlog.configure(wrapper_class=structlog.make_filtering_bound_logger(logging.WARNING))

from fsspec.implementations.dirfs import DirFileSystem

from toop_engine_interfaces.folder_structure import PREPROCESSING_PATHS
from toop_engine_interfaces.messages.preprocess.preprocess_commands import (
    AreaSettings,
    CgmesImporterParameters,
    PreprocessParameters,
)
from toop_engine_interfaces.nminus1_definition import load_nminus1_definition

FIXTURE = Path("../data/complex_grid")
GRID_FILE = FIXTURE / "grid.xiidm"

# True: AC-validate against the input N-1 definition of this grid, validated against the processed grid.
# False: AC-validate against the N-1 definition the importer derives from the grid masks.
USE_INPUT_NMINUS1_DEFINITION = True

# The importer writes into data_folder; keep the checked-in fixture read-only.
WORK = Path(tempfile.mkdtemp(prefix="toop_example5_"))
print("grid       :", GRID_FILE)
print("work folder:", WORK)


## Stage 1 — Importer

`CgmesImporterParameters` is the input contract. With **no** contingency file, the importer derives the
masks and the N-1 definition from the grid itself, including one contingency per switch in the N-1 area.
What we care about here is that this import also writes an `action_set` the optimizer can search.


In [ ]:
from toop_engine_importer.pypowsybl_import import preprocessing

importer_parameters = CgmesImporterParameters(
    grid_model_file=GRID_FILE,
    data_folder=WORK,
    area_settings=AreaSettings(
        cutoff_voltage=1.0,
        control_area=["BE", "NL"],
        view_area=["BE", "NL"],
        nminus1_area=["BE", "NL"],
    ),
)

import_result = preprocessing.convert_file(importer_parameters=importer_parameters)
print(type(import_result).__name__, "->", sorted(p.name for p in WORK.iterdir()))

mask_derived_nminus1 = load_nminus1_definition(WORK / PREPROCESSING_PATHS["nminus1_definition_file_path"])
print(f"monitored elements: {len(mask_derived_nminus1.monitored_elements)}")
print(f"contingencies     : {len(mask_derived_nminus1.contingencies)}")


### Choosing the N-1 definition for AC

The input N-1 definition is an `Nminus1Definition` with Powsybl ids. `filter_nminus1_definition_to_network`
turns it into a **grid-validated** N-1 definition: elements the grid does not contain are dropped and element
types are aligned with the grid, each change logged as a warning. We validate against the processed grid,
because that is the grid AC runs on. The import replaced the three-winding transformers by their legs, so the
transformers themselves are dropped here: from the contingencies, the monitored elements and the SPPS rule of
`C_3W`. Their contingencies keep only the surrounding switches.

DC does not read this definition: it optimizes against the outages derived from the masks. The choice only
decides which N-1 cases the AC validation in Stage 4 runs.

In [ ]:
from toop_engine_grid_helpers.powsybl.example_grids import create_complex_grid_nminus1_definition
from toop_engine_importer.pypowsybl_import.contingency_from_file import filter_nminus1_definition_to_network

if USE_INPUT_NMINUS1_DEFINITION:
    processed_grid = pypowsybl.network.load(str(WORK / PREPROCESSING_PATHS["grid_file_path_powsybl"]))
    ac_nminus1_definition = filter_nminus1_definition_to_network(create_complex_grid_nminus1_definition(), processed_grid)
    print("AC N-1 definition: input N-1 definition, grid-validated")
else:
    ac_nminus1_definition = mask_derived_nminus1
    print("AC N-1 definition: mask-derived")
print(f"contingencies     : {[contingency.id for contingency in ac_nminus1_definition.contingencies]}")

## Stage 2 — DC preprocessing

`load_grid` builds the solver inputs from the processed grid and its masks — `StaticInformation` (which
carries the `action_set`) and `NetworkData`. It logs a line for every element it cannot represent in DC;
we quiet that to `ERROR` for the duration of the call.


In [ ]:
from toop_engine_dc_solver.preprocess.convert_to_jax import load_grid

structlog.configure(wrapper_class=structlog.make_filtering_bound_logger(logging.ERROR))
stats, static_information, network_data = load_grid(
    data_folder_dirfs=DirFileSystem(str(WORK)),
    pandapower=False,
    parameters=PreprocessParameters(action_set_clip=2**10),
)
structlog.configure(wrapper_class=structlog.make_filtering_bound_logger(logging.WARNING))

print("solver contingencies   :", network_data.contingency_ids)
print("actionable substations :",
      len(static_information.dynamic_information.action_set.action_start_indices))


## Stage 3 — Topology optimization (DC)

`initialize_optimization` evaluates the unsplit baseline; each `run_epoch` consumes one chunk of the
candidate workset. We loop until the workset is exhausted (with a wall-clock guard), then take the best of
the improved topologies. Its `actions` and `disconnections` are indices into the same `action_set` the
import wrote — that is the object we replay onto the grid in Stage 4.


In [ ]:
from toop_engine_topology_optimizer.dc_bruteforce.optimizer import (
    get_num_branch_topologies_tried,
    initialize_optimization,
    is_exhausted,
    run_epoch,
)
from toop_engine_topology_optimizer.interfaces.messages.commons import Framework, GridFile
from toop_engine_topology_optimizer.interfaces.messages.dc_params import (
    BatchedMEParameters,
    DCOptimizerParameters,
    DescriptorDef,
    LoadflowSolverParameters,
)

grid_file = GridFile(framework=Framework.PYPOWSYBL, grid_folder=WORK.name)
dc_params = DCOptimizerParameters(
    summary_frequency=1,
    check_command_frequency=1,
    ga_config=BatchedMEParameters(
        runtime_seconds=60,
        iterations_per_epoch=8,
        me_descriptors=(DescriptorDef(metric="split_subs", num_cells=2),),
    ),
    loadflow_solver_config=LoadflowSolverParameters(max_num_splits=1, max_num_disconnections=0),
)

optimizer_data, _dynamic_stats, _initial_strategy = initialize_optimization(
    params=dc_params,
    optimization_id="example5",
    static_information_files=[grid_file.static_information_file],
    processed_gridfile_fs=DirFileSystem(str(WORK.parent)),
)

budget_seconds = 120.0
started = time.monotonic()
epochs, improved = 0, []
while not is_exhausted(optimizer_data) and time.monotonic() - started < budget_seconds:
    optimizer_data = run_epoch(optimizer_data)
    epochs += 1
    improved.extend(optimizer_data.latest_topologies)

print(f"epochs run   : {epochs}")
print(f"evaluated    : {get_num_branch_topologies_tried(optimizer_data)}"
      f"/{optimizer_data.runtime_state.total_workset_size}  (exhausted={is_exhausted(optimizer_data)})")
print(f"baseline DC fitness : {optimizer_data.initial_fitness:.4f}")

if improved:
    best = max(improved, key=lambda topology: topology.metrics.fitness)
    print(f"\nbest topology : actions={best.actions}  disconnections={best.disconnections}")
    print(f"best DC fitness: {best.metrics.fitness:.4f}  (baseline {optimizer_data.initial_fitness:.4f})")
else:
    best = None
    print("\nno improved topology found -- AC validation below covers the initial topology only")


## Stage 4 — AC validation of both topologies

The optimizer works in DC. To trust a result we replay it in AC and run the full N-1 analysis.

Two direct calls do it:

- `apply_topology` / `apply_disconnections` (`toop_engine_dc_solver`) mutate a fresh copy of the grid to
  the requested topology, addressed by the `action_set`.
- `get_ac_loadflow_results` (`toop_engine_contingency_analysis`) runs the AC N-1 against the N-1 definition
  chosen in Stage 1: the **grid-validated** input N-1 definition, or the mask-derived one.

We run it twice: once with no actions (the **initial** topology) and once with the found topology.


In [ ]:
from toop_engine_contingency_analysis.ac_loadflow_service import get_ac_loadflow_results
from toop_engine_dc_solver.postprocess.postprocess_powsybl import (
    apply_disconnections as powsybl_apply_disconnections,
)
from toop_engine_dc_solver.postprocess.postprocess_powsybl import (
    apply_topology as powsybl_apply_topology,
)
from toop_engine_grid_helpers.powsybl.loadflow_parameters import CGMES_DISTRIBUTED_SLACK
from toop_engine_interfaces.stored_action_set import load_action_set

action_set = load_action_set(
    WORK / PREPROCESSING_PATHS["action_set_file_path"],
    WORK / PREPROCESSING_PATHS["action_set_diff_path"],
)
grid_on_disk = WORK / PREPROCESSING_PATHS["grid_file_path_powsybl"]


def ac_converged_frame(actions, disconnections):
    """Replay a topology onto a fresh grid copy and return the collected AC convergence frame."""
    net = pypowsybl.network.load(str(grid_on_disk))
    if actions:
        powsybl_apply_topology(net=net, actions=actions, action_set=action_set)
    if disconnections:
        powsybl_apply_disconnections(net, disconnections=disconnections, action_set=action_set)
    results = get_ac_loadflow_results(
        net=net,
        n_minus_1_definition=ac_nminus1_definition,
        timestep=0,
        lf_params=CGMES_DISTRIBUTED_SLACK,
    )
    converged = results.converged
    return converged.collect() if hasattr(converged, "collect") else converged


initial_converged = ac_converged_frame([], [])
found_converged = ac_converged_frame(best.actions, best.disconnections) if best is not None else None

print("initial topology :", initial_converged.height, "AC N-1 cases evaluated")
if found_converged is not None:
    print("found topology   :", found_converged.height, "AC N-1 cases evaluated")


### Did every loadflow converge?

A topology is only usable if **every** N-1 case converges. `status == "CONVERGED"` is the pass mark; any
`MAX_ITERATION_REACHED`, `FAILED`, or `NO_CALCULATION` is a case that did not solve. We reduce each
topology's convergence frame to that verdict, then list the offenders where there are any.


In [ ]:
def summarise(converged: pl.DataFrame, label: str) -> dict:
    total = converged.height
    n_converged = converged.filter(pl.col("status") == "CONVERGED").height
    non_converged = (
        converged.filter(pl.col("status") != "CONVERGED")
        .select("contingency", "status")
        .to_dicts()
    )
    return {
        "topology": label,
        "cases": total,
        "converged": n_converged,
        "all_converged": total == n_converged,
        "non_converged": non_converged,
    }


rows = [summarise(initial_converged, "initial")]
if found_converged is not None:
    rows.append(summarise(found_converged, f"found actions={best.actions}"))

summary = pl.DataFrame(
    [{k: row[k] for k in ("topology", "cases", "converged", "all_converged")} for row in rows]
)
print(summary)

for row in rows:
    verdict = "ALL CONVERGED" if row["all_converged"] else "NOT all converged"
    print(f"\n{row['topology']:<24} {verdict}  ({row['converged']}/{row['cases']})")
    for case in row["non_converged"]:
        print(f"    - {case['contingency']}: {case['status']}")


## Recap

- The **importer** writes the processed grid, the masks, the `action_set` and the mask-derived
  `nminus1_definition.json`. An input N-1 definition is turned into a **grid-validated** N-1 definition with
  `filter_nminus1_definition_to_network`.
- **DC** turns that into solver inputs and searches the action set for a better topology, returning it as
  plain `actions` / `disconnections` indices.
- **CA/AC** replays those indices onto a fresh grid copy with `apply_topology` and runs the full N-1 AC
  with `get_ac_loadflow_results` against the chosen N-1 definition, persisting nothing.

The convergence table is the payoff: the same set of real N-1 cases run against the initial and the found
topology, each answered with a single "did every loadflow converge?" verdict — which is what tells you
whether the optimizer's DC-found topology is actually feasible in AC.
